In [3]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
from sklearn.model_selection import RandomizedSearchCV

# 1. โหลดข้อมูล
train_df = pd.read_csv("./cleaned_data/train_data.csv")
test_df = pd.read_csv("./cleaned_data/test_data.csv")

# ==========================================
# 2. FEATURE ENGINEERING FUNCTION
# ==========================================
def engineer_features(df):
    df = df.copy()
    
    # 1) จัดกลุ่มอายุ (เน้นกลุ่มวัยเกษียณ)
    df['is_senior'] = (df['age'] >= 60).astype(int)
    
    # 2) สถานะความพร้อมทางการเงิน (มีเงินเหลือ และ ไม่มีหนี้)
    df['has_free_cash'] = ((df['balance'] > 0) & (df['housing'] == 'no') & (df['loan'] == 'no')).astype(int)
    
    # 3) เคยถูกติดต่อในแคมเปญอดีตหรือไม่ (pdays != -1 หรือ 999 แล้วแต่การ clean)
    df['was_previously_contacted'] = (df['pdays'] != -1).astype(int)
    
    # 4) ความถี่รวมในการติดต่อทั้งหมด
    df['total_contacts'] = df['campaign'] + df['previous']
    
    # 5) ยอดเงินติดลบหรือไม่
    df['is_balance_negative'] = (df['balance'] < 0).astype(int)
    
    return df

# ปรับใช้ Feature Engineering กับทั้ง Train และ Test
train_fe = engineer_features(train_df)
test_fe = engineer_features(test_df)

# 3. แยก X และ y
X_train = train_fe.drop(columns=['y'])
y_train = train_fe['y']
X_test = test_fe.drop(columns=['y'])
y_test = test_fe['y']

# 4. One-Hot Encoding
X_train_encoded = pd.get_dummies(X_train, drop_first=True)
X_test_encoded = pd.get_dummies(X_test, drop_first=True)
X_train_encoded, X_test_encoded = X_train_encoded.align(X_test_encoded, join='left', axis=1, fill_value=0)

print(f"จำนวน Features หลังทำ FE: {X_train_encoded.shape[1]} คอลัมน์")

# 5. จูน Hyperparameter ด้วย RandomizedSearchCV
param_dist = {
    'n_estimators': [100, 200, 300],
    'max_depth': [10, 15, 20],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [2, 5, 10]
}

rf_base = RandomForestClassifier(random_state=0, class_weight='balanced')

random_search = RandomizedSearchCV(
    estimator=rf_base, 
    param_distributions=param_dist, 
    n_iter=10, 
    cv=3, 
    scoring='f1', 
    random_state=0, 
    n_jobs=-1
)

print("กำลังเทรนโมเดลพร้อม Features ใหม่...")
random_search.fit(X_train_encoded, y_train)

best_rf_model = random_search.best_estimator_

# 6. ประเมินผล
y_pred = best_rf_model.predict(X_test_encoded)

print("\n==========================================")
print("Accuracy Score:", accuracy_score(y_test, y_pred))
print("==========================================")
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

# 7. ดู Feature Importances รวมฟีเจอร์ใหม่
importances = pd.DataFrame({
    'Feature': X_train_encoded.columns,
    'Importance': best_rf_model.feature_importances_
}).sort_values(by='Importance', ascending=False)

print("\nTop 10 Most Important Features:\n", importances.head(10))

จำนวน Features หลังทำ FE: 44 คอลัมน์
กำลังเทรนโมเดลพร้อม Features ใหม่...

Accuracy Score: 0.8472045375622178

Confusion Matrix:
 [[6753  898]
 [ 422  566]]

Classification Report:
               precision    recall  f1-score   support

           0       0.94      0.88      0.91      7651
           1       0.39      0.57      0.46       988

    accuracy                           0.85      8639
   macro avg       0.66      0.73      0.69      8639
weighted avg       0.88      0.85      0.86      8639


Top 10 Most Important Features:
                Feature  Importance
1              balance    0.112405
0                  age    0.103516
2                  day    0.096435
30  contact_unrecorded    0.064492
43    poutcome_success    0.060209
4                pdays    0.051137
10      total_contacts    0.042879
3             campaign    0.042760
8        has_free_cash    0.042037
27         housing_yes    0.029241
